# Member 1 — Data & Pipeline
**Dengue Cluster Early-Warning · Gen AI Academy APAC**

Owns: ingest → clean → grid → BigQuery. Delivers the tables every other member builds on.

**Deliverables (the data contract):**
1. `dengue_ew.telemetry_daily` — cell_id, lat, lon, date, case_count, rain_mm
2. `dengue_ew.features` — adds case_density_14d, rain_lag_7to14d, recurrence
3. `handoff/features_latest.csv` — same as (2), latest date only, for anyone not on BigQuery yet

> Runtime: normal CPU is fine for Member 1. Member 2 needs the GPU.

In [ ]:
# -- 0. Setup -------------------------------------------------------
import os, io, json, zipfile, urllib.request
import numpy as np, pandas as pd
os.makedirs('data', exist_ok=True); os.makedirs('handoff', exist_ok=True)
print(pd.__version__)


## 1. Source A — NEA active dengue clusters (today's snapshot)
Official data.gov.sg poll-download flow. Gives **active clusters now**: polygons + case counts.
Snapshot only — no history — so Source B supplies the time series.

In [ ]:
# -- 1. NEA active clusters (GeoJSON via poll-download) -------------
import requests

DATASET_ID = "d_dbfabf16158d1b0e1c420627c0819168"   # Dengue Clusters (GEOJSON), NEA
clusters_now = None
try:
    meta = requests.get(f"https://api-open.data.gov.sg/v1/public/api/datasets/{DATASET_ID}/poll-download", timeout=30).json()
    assert meta.get('code') == 0, meta.get('errMsg')
    gj = requests.get(meta['data']['url'], timeout=60).json()
    rows = []
    for f in gj.get('features', []):
        props = f.get('properties', {})
        geom = f.get('geometry', {})
        coords = geom.get('coordinates', [])
        # polygon centroid (rough): mean of ring vertices
        if geom.get('type') == 'Polygon' and coords:
            ring = np.array(coords[0])[:, :2]
            lon, lat = ring[:,0].mean(), ring[:,1].mean()
            rows.append({'lat': lat, 'lon': lon,
                         'case_count': props.get('CASE_SIZE') or props.get('case_size') or np.nan,
                         'locality': str(props.get('LOCALITY') or props.get('locality') or '')[:120]})
    clusters_now = pd.DataFrame(rows)
    clusters_now['snapshot_date'] = pd.Timestamp.today().normalize()
    clusters_now.to_csv('data/nea_clusters_today.csv', index=False)
    print(f"Active clusters today: {len(clusters_now)}")
except Exception as e:
    print("NEA snapshot fetch failed (fine for now, Source B carries history):", e)
clusters_now.head() if clusters_now is not None else None


## 2. Source B — historical cluster snapshots (time series)
SGCharts Outbreak archive: NEA webpage snapshots since 2013 as CSVs with lat/lon, case counts, dates.
**Attribution required:** National Environment Agency (nea.gov.sg/dengue-zika/dengue/dengue-clusters).

Manual step (5 min): download a recent batch of cluster CSVs from outbreak.sgcharts.com/data into `data/history/`.
The loader below reads whatever it finds. If empty, the synthetic fallback keeps the pipeline testable so no teammate is blocked.

In [ ]:
# -- 2. Load historical snapshots (or synthetic fallback) -----------
import glob
hist_files = glob.glob('data/history/*.csv')
rng = np.random.default_rng(7)

def load_history(files):
    frames = []
    for f in files:
        try:
            h = pd.read_csv(f)
            h.columns = [c.strip().lower().replace(' ', '_') for c in h.columns]
            lat = next((c for c in h.columns if 'lat' in c), None)
            lon = next((c for c in h.columns if 'lon' in c or 'lng' in c), None)
            cc  = next((c for c in h.columns if 'case' in c), None)
            dt  = next((c for c in h.columns if 'date' in c), None)
            if not all([lat, lon, cc, dt]): continue
            h = h[[lat, lon, cc, dt]].rename(columns={lat:'lat', lon:'lon', cc:'case_count', dt:'date'})
            h['date'] = pd.to_datetime(h['date'].astype(str), format='%y%m%d', errors='coerce')
            frames.append(h)
        except Exception as e:
            print('skip', f, e)
    return pd.concat(frames, ignore_index=True) if frames else None

def synthetic_history(n_days=180):
    frames = []
    hotspots = [(1.35+rng.uniform(-.08,.08), 103.85+rng.uniform(-.15,.15), rng.gamma(2,2)) for _ in range(40)]
    for d in pd.date_range(end=pd.Timestamp.today().normalize(), periods=n_days, freq='D'):
        for (la, lo, s) in hotspots:
            if rng.random() < 0.6:
                frames.append({'lat': la+rng.normal(0,.004), 'lon': lo+rng.normal(0,.004),
                               'case_count': max(1, int(rng.poisson(s))), 'date': d})
    return pd.DataFrame(frames)

hist = load_history(hist_files)
if hist is None or hist.empty:
    print('No history CSVs found -> SYNTHETIC fallback (replace before final run!)')
    hist = synthetic_history()
hist = hist.dropna(subset=['lat','lon','date'])
hist['case_count'] = pd.to_numeric(hist['case_count'], errors='coerce').clip(lower=0)
hist = hist.dropna(subset=['case_count'])
hist = hist[(hist.lat.between(1.15, 1.50)) & (hist.lon.between(103.5, 104.15))]   # SG bounds sanity
print(f"History: {len(hist):,} rows, {hist.date.min().date()} -> {hist.date.max().date()}")


## 3. Source C — rainfall
data.gov.sg realtime rainfall API gives per-station readings. NEA has flagged station network issues,
so: retries, drop NaNs, and a monsoon-shaped synthetic fallback so downstream never stalls.

In [ ]:
# -- 3. Rainfall (daily island-mean; API with fallback) --------------
def fetch_rain(dates):
    vals = {}
    for d in dates:
        try:
            r = requests.get('https://api-open.data.gov.sg/v2/real-time/api/rainfall',
                             params={'date': d.strftime('%Y-%m-%d')}, timeout=20).json()
            readings = r.get('data', {}).get('readings', [])
            day = [it.get('value') for rd in readings for it in rd.get('data', []) if it.get('value') is not None]
            if day: vals[d] = float(np.mean(day)) * 24   # mean mm per reading -> rough daily mm
        except Exception:
            pass
    return vals

dates = pd.date_range(hist.date.min(), hist.date.max(), freq='D')
rain_vals = {}                      # set to fetch_rain(dates) when online + patient (1 call/day)
if not rain_vals:
    t = np.arange(len(dates))
    rain_vals = dict(zip(dates, np.clip(rng.gamma(2, 5, len(dates)) * (1 + 0.5*np.sin(2*np.pi*t/120)), 0, None)))
    print('Using synthetic rainfall (swap in fetch_rain for final run)')
rain = pd.DataFrame({'date': list(rain_vals.keys()), 'rain_mm': list(rain_vals.values())})
print(rain.describe().round(1))


## 4. Grid assignment + daily cell table → the contract table #1

In [ ]:
# -- 4. 250m-ish grid, daily aggregation ------------------------------
STEP = 0.00225   # ~250 m
hist['cell_lat'] = (hist['lat'] // STEP * STEP).round(5)
hist['cell_lon'] = (hist['lon'] // STEP * STEP).round(5)
hist['cell_id']  = hist['cell_lat'].astype(str) + '_' + hist['cell_lon'].astype(str)

daily = (hist.groupby(['cell_id','cell_lat','cell_lon','date'], as_index=False)
             .agg(case_count=('case_count','sum'))
             .rename(columns={'cell_lat':'lat','cell_lon':'lon'}))

# densify: every active cell gets every date (zeros matter for rolling windows)
cells = daily[['cell_id','lat','lon']].drop_duplicates()
full = (cells.assign(k=1).merge(pd.DataFrame({'date': dates, 'k':1}), on='k').drop(columns='k')
             .merge(daily, on=['cell_id','lat','lon','date'], how='left'))
full['case_count'] = full['case_count'].fillna(0)
full = full.merge(rain, on='date', how='left')
full['rain_mm'] = full['rain_mm'].ffill().fillna(0)

print(f"telemetry_daily: {full.cell_id.nunique():,} cells x {full.date.nunique()} days = {len(full):,} rows")
full.to_csv('handoff/telemetry_daily.csv', index=False)
full.head()


## 5. Rolling features → contract table #2 (Member 2's input)

In [ ]:
# -- 5. Feature engineering -------------------------------------------
d = full.sort_values(['cell_id','date']).reset_index(drop=True)
g = d.groupby('cell_id')
d['case_density_14d']    = g['case_count'].transform(lambda s: s.rolling(14, min_periods=7).sum())
d['rain_lag_7to14d']     = g['rain_mm'].transform(lambda s: s.rolling(14, min_periods=7).mean().shift(7))
d['recurrence']          = g['case_count'].transform(lambda s: s.expanding().mean())
features = d.dropna(subset=['case_density_14d'])

latest = features[features.date == features.date.max()]
latest.to_csv('handoff/features_latest.csv', index=False)
features.to_csv('handoff/features_full.csv', index=False)
print(f"features: {len(features):,} rows · latest snapshot: {len(latest):,} cells -> handoff/features_latest.csv")
latest.nlargest(10, 'case_density_14d')[['cell_id','lat','lon','case_density_14d']]


## 6. Upload to BigQuery (the official handoff)
Creates `dengue_ew.telemetry_daily` and `dengue_ew.features`. Member 2 reads `features`;
Member 3 points Looker Studio at whatever Member 2 writes back (`inspection_priority`).

In [ ]:
# -- 6. BigQuery load --------------------------------------------------
RUN_BQ = False                      # flip True with the team GCP project
PROJECT = "your-team-project-id"

if RUN_BQ:
    from google.colab import auth; auth.authenticate_user()
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT)
    client.create_dataset(f"{PROJECT}.dengue_ew", exists_ok=True)
    for name, frame in [("telemetry_daily", full), ("features", features)]:
        job = client.load_table_from_dataframe(frame, f"{PROJECT}.dengue_ew.{name}",
              job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
        job.result(); print(name, "->", job.output_rows, "rows")


---
## Data contract (paste into team chat)

| Table / file | Owner | Consumer | Columns |
|---|---|---|---|
| `dengue_ew.telemetry_daily` | M1 | M2 | cell_id, lat, lon, date, case_count, rain_mm |
| `dengue_ew.features` (+ features_latest.csv) | M1 | M2 | above + case_density_14d, rain_lag_7to14d, recurrence |
| `dengue_ew.inspection_priority` | M2 | M3, M4 | rank, cell_id, lat, lon, risk_score, top factor components |

Attribution line for the deck: *Dengue cluster data © National Environment Agency, via data.gov.sg and outbreak.sgcharts.com archive.*